In [1]:
from pdb import set_trace as st
from pprint import pprint
import json
import subprocess
import sys

import msgspec
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import os
import re
import torch
from sentence_transformers import models, SentenceTransformer
from transformers import AutoTokenizer, AutoModel, HfArgumentParser
from tevatron.retriever.arguments import DataArguments, ModelArguments
from tevatron.retriever.arguments import TevatronTrainingArguments as TrainingArguments
from tevatron.retriever.modeling import DenseModel

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

/scratch/ft49/thuy0050/miniconda/conda/envs/tevatron/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Utils


In [2]:
def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()

    with open(file_path, "rb") as file:
        data = file.read()
    if jsonl:
        output = decoder.decode_lines(data)
    else:
        output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(encoder.encode(data))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)


def read_tsv(file_path, row_names=None):
    file_path = Path(file_path)
    if file_path.is_file() and file_path.suffix == ".tsv" :
        temp = pd.read_csv(file_path, sep='\t', names=row_names)
    else:
        raise ValueError("filepath is not a file or it is not a tsv file.")
    return temp


# Compute token embeddings
def get_sentence_embeddings(data_args, model, tokenizer, q=None, p=None):
    
    if q is not None:
        if not isinstance(q, list):
            q = [q]
        q = tokenizer(
            q,
            padding=True,
            truncation=True,
            max_length=(
                data_args.query_max_len - 1
                if data_args.append_eos_token
                else data_args.query_max_len
            ),
            pad_to_multiple_of=data_args.pad_to_multiple_of,
            return_attention_mask=True,
            return_tensors="pt",
            return_token_type_ids=False,
            add_special_tokens=True,
        )
        for k, v in q.items():
            q[k] = v.to("cuda")
    elif p is not None:
        if not isinstance(p, list):
            p = [p]
        p = tokenizer(
            p,
            padding=True,
            truncation=True,
            max_length=(
                data_args.passage_max_len - 1
                if data_args.append_eos_token
                else data_args.passage_max_len
            ),
            pad_to_multiple_of=data_args.pad_to_multiple_of,
            return_attention_mask=True,
            return_tensors="pt",
            return_token_type_ids=False,
            add_special_tokens=True,
        )
        for k, v in p.items():
            p[k] = v.to("cuda")
    
    model.eval()
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        with torch.no_grad():
            if q is not None:
                output = model(query=q).q_reps
            else:
                output = model(passage=p).p_reps
    return output


def get_cosine_similarity(a, b, normalize=True):
    # This assume that the embedding has been normalized
    
    if normalize:
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
    
    return torch.matmul(a, b.T)

In [3]:
# bsz : batch size (number of positive pairs)
# d   : latent dim
# x   : Tensor, shape=[bsz, d]
#       latents for one side of positive pairs
# y   : Tensor, shape=[bsz, d]
#       latents for the other side of positive pairs

def align_loss(x, y, alpha=2):
    return (x - y).norm(p=2, dim=1).pow(alpha).mean()

def uniform_loss(x, t=2):
    return torch.pdist(x, p=2).pow(2).mul(-t).exp().mean().log()

def get_info(q_pos, q_neg, p, model_list, tokenizer, model_checkpoint_dir_list, data_args):
    
    for ix, (model, model_checkpoint_dir) in enumerate(zip(model_list, model_checkpoint_dir_list)):
        p1 = get_sentence_embeddings(data_args, model, tokenizer, p=p)
        q_pos1 = get_sentence_embeddings(data_args, model, tokenizer, q=q_pos)
        q_neg1 = get_sentence_embeddings(data_args, model, tokenizer, q=q_neg)
        
        print(f"\nModel{ix+1}:", Path(model_checkpoint_dir).stem)
        print(f"sim(p, q+): {get_cosine_similarity(p1, q_pos1).mean()}\nsim(p, q-): {get_cosine_similarity(p1, q_neg1).mean()}\nalign_loss(p, q+): {align_loss(p1, q_pos1).mean()}\nalign_loss(p, q-): {align_loss(p1, q_neg1).mean()}\ntemporal sim(p, q+): {get_cosine_similarity(p1[:, 512:], q_pos1[:, 512:]).mean()}\ntemporal sim(p, q-): {get_cosine_similarity(p1[:, 512:], q_neg1[:, 512:]).mean()}\nuniformity: {uniform_loss(torch.concat([p1, q_pos1, q_neg1], dim=0))}\n")
        # print(f"sim(p, q+): {get_cosine_similarity(p1, q_pos1)}\nsim(p, q-): {get_cosine_similarity(p1, q_neg1)}\nalign_loss(p, q+): {align_loss(p1, q_pos1)}\nalign_loss(p, q-): {align_loss(p1, q_neg1)}\ntemporal sim(p, q+): {get_cosine_similarity(p1[:, 512:], q_pos1[:, 512:])}\ntemporal sim(p, q-): {get_cosine_similarity(p1[:, 512:], q_neg1[:, 512:])}")
    
    # p2 = get_sentence_embeddings(data_args, model2, tokenizer, p=p)
    # q_pos2 = get_sentence_embeddings(data_args, model2, tokenizer, q=q_pos)
    # q_neg2 = get_sentence_embeddings(data_args, model2, tokenizer, q=q_neg)
    
    # print("Model2:", Path(model2_checkpoint_dir).stem)
    # # print(f"sim(p, q+): {get_cosine_similarity(p2, q_pos2).mean()}\nsim(p, q-): {get_cosine_similarity(p2, q_neg2). mean()}\nalign_loss(p, q+): {align_loss(p2, q_pos2).mean()}\nalign_loss(p, q-): {align_loss(p2, q_neg2).mean()}\ntemporal sim(p, q+): {get_cosine_similarity(p2[:, 512:], q_pos2[:, 512:]).mean()}\ntemporal sim(p, q-): {get_cosine_similarity(p2[:, 512:], q_neg2[:, 512:]).mean()}\nuniformity: {uniform_loss(torch.concat([p2, q_pos2, q_neg2], dim=0))}")
    # print(f"sim(p, q+): {get_cosine_similarity(p2, q_pos2)}\nsim(p, q-): {get_cosine_similarity(p2, q_neg2)}\nalign_loss(p, q+): {align_loss(p2, q_pos2)}\nalign_loss(p, q-): {align_loss(p2, q_neg2)}\ntemporal sim(p, q+): {get_cosine_similarity(p2[:, 512:], q_pos2[:, 512:])}\ntemporal sim(p, q-): {get_cosine_similarity(p2[:, 512:], q_neg2[:, 512:])}")
    

# Load models

In [7]:
os.listdir("/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/ts-retriever/contriever/bs64")

['baseline',
 'semantic_matryoshka',
 'temporal_projector_reconstruction_kl_loss',
 'temporal_projector_reconstruction',
 'temporal_projector',
 'temporal']

In [8]:
DATA_ROOT_DIR="/home/thuy0050/mg61_scratch2/thuy0050/data/third_work"
OUTPUT_DIR_ROOT="/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron"

DATA_NAME="temporal_nobel_prize"
MODEL_NAME="ts-retriever"
BACKBONE="bge-base-en-v1.5"
EXP_NAME = ["BAAI/bge-base-en-v1.5", "baseline", "semantic_matryoshka", "temporal", "temporal_projector", "temporal_projector_reconstruction", "temporal_projector_reconstruction_kl_loss"]
model_list = []
model_checkpoint_dir_list = []
torch_dtype = torch.bfloat16

for ix, exp in enumerate(EXP_NAME):
    
    if ix == 0:
        OUTPUT_DIR = exp
    else:
        OUTPUT_DIR=f"{OUTPUT_DIR_ROOT}/{DATA_NAME}/{MODEL_NAME}/{BACKBONE}/{exp}"
    CHECKPOINT_DIR=OUTPUT_DIR

    sys.argv = [
        "train_tsretriever_with_temporal_v4.py",  # dummy script name
        "--pooling", "avg",
        "--bf16",
        "--normalize",
        "--query_max_len", "512",
        "--passage_max_len", "512",
        "--attn_implementation", "sdpa",
        "--lora",
        "--lora_r", "4",
        "--lora_alpha", "16",
        "--lora_target_modules", "all-linear",
        "--modules_to_save", "temporal_projector",
        "--dataset_name", f"{DATA_ROOT_DIR}/tevatron/Tevatron___msmarco-passage",
        "--dataset_path", f"{DATA_ROOT_DIR}/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl",
        "--eval_dataset_path", f"{DATA_ROOT_DIR}/temporal/temporal_nobel_prize/train/dev.jsonl",
        "--model_name_or_path", CHECKPOINT_DIR,
        "--run_name", f"{BACKBONE}_{EXP_NAME}"
    ]


    parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

    model_args, data_args, training_args = parser.parse_args_into_dataclasses()
    model_args: ModelArguments
    data_args: DataArguments
    training_args: TrainingArguments

    tokenizer = AutoTokenizer.from_pretrained(model_args.model_name_or_path)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    if data_args.padding_side == 'right':
        tokenizer.padding_side = 'right'
    else:
        tokenizer.padding_side = 'left'
        
    model = DenseModel.load(
        model_args.model_name_or_path,
        pooling=model_args.pooling,
        normalize=model_args.normalize,
        lora_name_or_path=model_args.lora_name_or_path,
        cache_dir=model_args.cache_dir,
        torch_dtype=torch_dtype,
        attn_implementation=model_args.attn_implementation,
    )
    model = model.to("cuda")
    model_list.append(model)
    model_checkpoint_dir_list.append(CHECKPOINT_DIR)

Either your model is not a PEFT-model or you are missing the lora_name_or_path argument.
BertModel does not support Flash Attention 2.0 yet. Please request to add support where the model is hosted, on its model hub page: https://huggingface.co/BAAI/bge-base-en-v1.5/discussions/new or in the Transformers GitHub repo: https://github.com/huggingface/transformers/issues/new
Traceback (most recent call last):
  File "/home/thuy0050/code/tevatron/src/tevatron/retriever/modeling/encoder.py", line 244, in try_using_flash_attn
    base_model = cls.TRANSFORMER_CLS.from_pretrained(
  File "/scratch/ft49/thuy0050/miniconda/conda/envs/tevatron/lib/python3.10/site-packages/transformers/models/auto/auto_factory.py", line 600, in from_pretrained
    return model_class.from_pretrained(
  File "/scratch/ft49/thuy0050/miniconda/conda/envs/tevatron/lib/python3.10/site-packages/transformers/modeling_utils.py", line 315, in _wrapper
    return func(*args, **kwargs)
  File "/scratch/ft49/thuy0050/miniconda/c

# Analysis

In [8]:
prompt = "Represent this sentence for searching relevant passages: "

dev = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/dev.jsonl", jsonl=True)

p_text = []
q_pos_text = []
q_neg_text = []
p_index = []
q_pos_count = []
q_neg_count = []

for ix, item in enumerate(dev):
    p_index.append(ix)
    p_text.append(item["query"])
    q_pos_count.append(len(item["positive_passages"]))
    q_neg_count.append(len(item["negative_passages"]))
    for i in item["positive_passages"]:
        q_pos_text.append(i['text'])
    for i in item["negative_passages"]:
        q_neg_text.append(i['text'])

# q_pos_text = prompt + "Jermaine Beckford played for which team from 2003 to 2004?"
# qt_pos_text = prompt + "from 2003 to 2004?"

# q_neg_text = prompt + "Jermaine Beckford played for which team from 1996 to 2002?"
# qt_neg_text = prompt + "from 1996 to 2002"

# p_text = "Beckford originally began his career in the Chelsea youth team , coming through the schoolboy ranks at the same time as Carlton Cole . Rejected by Chelsea in 2003 , he was signed up by Wealdstone , then in the Isthmian Premier League , and played as a semi-professional for three years whilst also working as a windscreen fitter for the RAC . His very impressive goal scoring record for Wealdstone attracted a lot of attention from Football League sides and reportedly more than 30 professional clubs showed an interest in the prolific striker , with many sending scouts to watch him play for Wealdstone . He had a trial with Championship side Crystal Palace , before signing for Leeds United in March 2006 for an undisclosed fee , having scored 35 goals in 40 games for Wealdstone that season ."

The file is of type: <class 'list'>
The file contains 165 items.


In [9]:
def get_alignment_score(q_pos, p, q_pos_count, p_index):
    align_score = 0.0
    temp = 0
    for k, v in zip(p_index, q_pos_count):
        align_score += align_loss(q_pos[temp:temp+v], p[k])
        temp += v
    return align_score / len(p_index)

def get_uniformity_score(p, q_pos, q_neg):
    return uniform_loss(torch.concat([p, q_pos, q_neg], dim=0))

## Test case

In [10]:
ix = 0
p_text[ix], q_pos_text[ix:ix+q_pos_count[ix]], q_neg_text[ix:ix+q_neg_count[ix]]
temp_p_text = "He signed a two-year contract with League One side Crewe Alexandra in June 2013 after manager Steve Davis paid Macclesfield an undisclosed fee .\
However , he played just five games for the Railwaymen , being sent on two loan spells to Lincoln City , before returning to Macclesfield Town in February 2015 .\
He then played for newly relegated Notts County in League Two for two seasons .\
In July 2017 , Audel joined Barrow , moving to Welling United a year later ."
print(temp_p_text.split("."))

['He signed a two-year contract with League One side Crewe Alexandra in June 2013 after manager Steve Davis paid Macclesfield an undisclosed fee ', 'However , he played just five games for the Railwaymen , being sent on two loan spells to Lincoln City , before returning to Macclesfield Town in February 2015 ', 'He then played for newly relegated Notts County in League Two for two seasons ', 'In July 2017 , Audel joined Barrow , moving to Welling United a year later ', '']


In [24]:
for i, (model, model_checkpoint_dir) in enumerate(zip(model_list, model_checkpoint_dir_list)):
    p1 = get_sentence_embeddings(data_args, model, tokenizer, p=temp_p_text)[:, 512:]
    sentence_list = q_pos_text[ix:ix+q_pos_count[ix]] + q_neg_text[ix:ix+q_neg_count[ix]]
    q_pos1 = get_sentence_embeddings(data_args, model_list[0], tokenizer, q=sentence_list)[:, 512:]
    print(f"\nModel{i+1}:", Path(model_checkpoint_dir).stem)
    print("Passage:", temp_p_text)
    for sent, sim in zip(sentence_list, get_cosine_similarity(p1, q_pos1).cpu().numpy().tolist()[0]):
        print(f"{sent}: {sim:.4f}")


Model1: bge-base-en-v1
Passage: He signed a two-year contract with League One side Crewe Alexandra in June 2013 after manager Steve Davis paid Macclesfield an undisclosed fee .However , he played just five games for the Railwaymen , being sent on two loan spells to Lincoln City , before returning to Macclesfield Town in February 2015 .He then played for newly relegated Notts County in League Two for two seasons .In July 2017 , Audel joined Barrow , moving to Welling United a year later .
Thierry Audel played for which team from 2013 to 2014?: 0.7441
Thierry Audel played for which team from 2014 to 2015?: 0.7282
Which team did the player Thierry Audel belong to between Dec 2013 and 2014?: 0.7312
Which team did the player Thierry Audel belong to between Jun 2014 and Dec 2014?: 0.7244
Thierry Audel played for which team from 2015 to 2019?: 0.7209
Thierry Audel played for which team from 2016 to 2024?: 0.7064
Which team did the player Thierry Audel belong to from 2009 to 2012?: 0.7206
Whi

### Controlled experiments

In [26]:
for ix, (model, model_checkpoint_dir) in enumerate(zip(model_list, model_checkpoint_dir_list)):
    p1 = get_sentence_embeddings(data_args, model, tokenizer, p=temp_p_text.split(".")[0])
    sentence_list = [
        f"Thierry Audel played for which team before {x}" for x in [str(i) for i in range(2013-5, 2013+5+1)]
    ]
    q_pos1 = get_sentence_embeddings(data_args, model_list[0], tokenizer, q=sentence_list)
    print(f"\nModel{ix+1}:", Path(model_checkpoint_dir).stem)
    print("Passage:", temp_p_text.split(".")[0])
    for sent, sim in zip(sentence_list, get_cosine_similarity(p1, q_pos1).cpu().numpy().tolist()[0]):
        print(f"{sent}: {sim:.4f}")


Model1: bge-base-en-v1
Passage: He signed a two-year contract with League One side Crewe Alexandra in June 2013 after manager Steve Davis paid Macclesfield an undisclosed fee 
Thierry Audel played for which team before 2008: 0.6174
Thierry Audel played for which team before 2009: 0.6121
Thierry Audel played for which team before 2010: 0.6123
Thierry Audel played for which team before 2011: 0.6244
Thierry Audel played for which team before 2012: 0.6297
Thierry Audel played for which team before 2013: 0.6515
Thierry Audel played for which team before 2014: 0.6332
Thierry Audel played for which team before 2015: 0.6143
Thierry Audel played for which team before 2016: 0.6202
Thierry Audel played for which team before 2017: 0.6167
Thierry Audel played for which team before 2018: 0.6171

Model2: baseline
Passage: He signed a two-year contract with League One side Crewe Alexandra in June 2013 after manager Steve Davis paid Macclesfield an undisclosed fee 
Thierry Audel played for which team 

### Get all the scores given p_list, q+_list, q-_list for all models

In [13]:
get_info(q_pos_text[ix:ix+q_pos_count[ix]], q_neg_text[ix:ix+q_neg_count[ix]], p_text[ix], model_list, tokenizer, model_checkpoint_dir_list, data_args)

/scratch/ft49/thuy0050/miniconda/conda/envs/tevatron/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



Model1: bge-base-en-v1
sim(p, q+): 0.4585000276565552
sim(p, q-): 0.46335381269454956
align_loss(p, q+): 1.0829999446868896
align_loss(p, q-): 1.0732924938201904
temporal sim(p, q+): 0.41364365816116333
temporal sim(p, q-): 0.4079062342643738
uniformity: -1.1863049268722534


Model2: baseline
sim(p, q+): 0.43228089809417725
sim(p, q-): 0.44116315245628357
align_loss(p, q+): 1.1354382038116455
align_loss(p, q-): 1.1176738739013672
temporal sim(p, q+): 0.3925715684890747
temporal sim(p, q-): 0.3913133144378662
uniformity: -1.2802460193634033


Model3: semantic_matryoshka
sim(p, q+): 0.3232511281967163
sim(p, q-): 0.33379101753234863
align_loss(p, q+): 1.353497862815857
align_loss(p, q-): 1.3324179649353027
temporal sim(p, q+): 0.28009629249572754
temporal sim(p, q-): 0.27985867857933044
uniformity: -1.4870426654815674


Model4: temporal
sim(p, q+): 0.32166239619255066
sim(p, q-): 0.3329392075538635
align_loss(p, q+): 1.356675148010254
align_loss(p, q-): 1.334121584892273
temporal sim(p,

### To get the uniformity scores correctly!

In [265]:
for ix, (model, model_checkpoint_dir) in enumerate(zip(model_list, model_checkpoint_dir_list)):
        print(f"\nModel{ix+1}:", Path(model_checkpoint_dir).stem)
        p1 = get_sentence_embeddings(data_args, model, tokenizer, p=p_text)
        q_pos1 = get_sentence_embeddings(data_args, model, tokenizer, q=q_pos_text)
        q_neg1 = get_sentence_embeddings(data_args, model, tokenizer, q=q_neg_text)
        
        print("Alignment score:", get_alignment_score(q_pos1, p1, q_pos_count, p_index).cpu().item())
        print("Uniformity score:", get_uniformity_score(q_pos1, p1, q_neg1).cpu().item())


Model1: bge-base-en-v1
Alignment score: 0.5486642122268677
Uniformity score: -1.8932181596755981

Model2: baseline
Alignment score: 0.4313362240791321
Uniformity score: -1.9493263959884644

Model3: semantic_matryoshka
Alignment score: 0.4841267764568329
Uniformity score: -2.3427894115448

Model4: temporal
Alignment score: 0.4822169840335846
Uniformity score: -2.3403773307800293

Model5: temporal_projector
Alignment score: 0.4711715579032898
Uniformity score: -2.3922383785247803

Model6: temporal_projector_reconstruction
Alignment score: 0.4147244989871979
Uniformity score: -1.972241759300232

Model7: temporal_projector_reconstruction_kl_loss
Alignment score: 0.4145970642566681
Uniformity score: -1.9710659980773926


In [28]:
temp = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/implicit_event_based/ComplexTempQA/ComplexTempQA_small.json", jsonl=True)

The file is of type: <class 'list'>
The file contains 23425567 items.


In [ ]:
temp[0]

{'id': 1,
 'question': 'Did the Winter Olympic Games in 1994 which had 1737 participants had a higher number of participants than the sports season in 2006 in Germany which had 32 participants?',
 'answer': ['yes'],
 'type': '2a',
 'rating': 1,
 'timeframe': ['1994-02-01', '2006-07-09'],
 'question_entity': ['9663', '37285'],
 'answer_entity': ['224013'],
 'question_country_entity': ['20', '183'],
 'is_unnamed': 1}

In [30]:
temp[1]

{'id': 2,
 'question': 'Did the Eurovision Song Contest edition in 2006 which had 37 participants had a higher number of participants than the Eurovision Song Contest edition in 2004 which had 36 participants?',
 'answer': ['yes'],
 'type': '2a',
 'rating': 1,
 'timeframe': ['2004-01-01', '2006-05-18'],
 'question_entity': ['10152', '10150'],
 'answer_entity': ['224013'],
 'question_country_entity': ['41', '43'],
 'is_unnamed': 1}

# Test bge-multilingual-gemma2

In [5]:
import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'<instruct>{task_description}\n<query>{query}'


task = 'Given a web search query, retrieve relevant passages that answer the query.'
queries = [
    get_detailed_instruct(task, 'how much protein should a female eat'),
    get_detailed_instruct(task, 'summit define')
]
# No need to add instructions for documents
documents = [
    "As a general guideline, the CDC's average requirement of protein for women ages 19 to 70 is 46 grams per day. But, as you can see from this chart, you'll need to increase that if you're expecting or training for a marathon. Check out the chart below to see how much protein you should be eating each day.",
    "Definition of summit for English Language Learners. : 1  the highest point of a mountain : the top of a mountain. : 2  the highest level. : 3  a meeting or series of meetings between the leaders of two or more governments."
]
input_texts = queries + documents

tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-multilingual-gemma2')
model = AutoModel.from_pretrained('BAAI/bge-multilingual-gemma2')
model.eval()

max_length = 4096
# Tokenize the input texts
batch_dict = tokenizer(input_texts, max_length=max_length, padding=True, truncation=True, return_tensors='pt', pad_to_multiple_of=8)

with torch.no_grad():
    outputs = model(**batch_dict)
    embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
    
# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)
scores = (embeddings[:2] @ embeddings[2:].T) * 100
print(scores.tolist())
# [[55.92064666748047, 1.6549524068832397], [-0.2698777914047241, 49.95653533935547]]


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 55.53it/s]


[[55.920654296875, 1.6549508571624756], [-0.26987946033477783, 49.95652389526367]]


In [51]:
queries = [get_detailed_instruct(task, x) for x in [
        "Thierry Audel played for which team after 2013?", 
        "Thierry Audel played for which team before 2013?", 
        "Thierry Audel played for which team in 2013?",
        "Thierry Audel played for which team as of 2013?",
        "Thierry Audel played for which team during 2013?",
    ]
]
documents = [
    f"He signed a two-year contract with League One side Crewe Alexandra in June {x} after manager Steve Davis paid Macclesfield an undisclosed fee" for x in [str(i) for i in range(2013-5, 2013+5+1)]
]

input_texts = queries + documents

max_length = 4096
# Tokenize the input texts
batch_dict = tokenizer(input_texts, max_length=max_length, padding=True, truncation=True, return_tensors='pt', pad_to_multiple_of=8)
model.cuda()

with torch.no_grad():
    outputs = model(**batch_dict.to("cuda"))
    embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
    
# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)

In [52]:
input_texts

['<instruct>Given a web search query, retrieve relevant passages that answer the query.\n<query>Thierry Audel played for which team after 2013?',
 '<instruct>Given a web search query, retrieve relevant passages that answer the query.\n<query>Thierry Audel played for which team before 2013?',
 '<instruct>Given a web search query, retrieve relevant passages that answer the query.\n<query>Thierry Audel played for which team in 2013?',
 '<instruct>Given a web search query, retrieve relevant passages that answer the query.\n<query>Thierry Audel played for which team as of 2013?',
 '<instruct>Given a web search query, retrieve relevant passages that answer the query.\n<query>Thierry Audel played for which team during 2013?',
 'He signed a two-year contract with League One side Crewe Alexandra in June 2008 after manager Steve Davis paid Macclesfield an undisclosed fee',
 'He signed a two-year contract with League One side Crewe Alexandra in June 2009 after manager Steve Davis paid Macclesfiel

In [54]:
get_cosine_similarity(embeddings[:5, :], embeddings[5:, :]).cpu().numpy().tolist()

[[0.1552232950925827,
  0.16177912056446075,
  0.15827026963233948,
  0.16438914835453033,
  0.18467970192432404,
  0.2394535392522812,
  0.20880185067653656,
  0.19272717833518982,
  0.17711526155471802,
  0.1718238741159439,
  0.16900385916233063],
 [0.1308995485305786,
  0.13681559264659882,
  0.1339738517999649,
  0.1398969441652298,
  0.15411652624607086,
  0.1984422504901886,
  0.1577611267566681,
  0.1390952318906784,
  0.12733295559883118,
  0.12302228063344955,
  0.11959163844585419],
 [0.12189722806215286,
  0.1305418461561203,
  0.13000202178955078,
  0.1399027705192566,
  0.15953484177589417,
  0.21812564134597778,
  0.17189544439315796,
  0.1497773826122284,
  0.13343586027622223,
  0.1285763531923294,
  0.12835368514060974],
 [0.12579186260700226,
  0.1331844925880432,
  0.13269543647766113,
  0.1414947360754013,
  0.15805937349796295,
  0.20887385308742523,
  0.16825731098651886,
  0.15186387300491333,
  0.13853007555007935,
  0.13379134237766266,
  0.13157813251018524],

In [58]:
sum([v.numel() for k, v in model.named_parameters()]) / 1024**2

8813.58447265625